# CE 310 — Week 13 In-Class Exercise: Monte Carlo Simulation
**Course:** Probability & Statistics for Civil/Architectural Engineering  
**University of Arizona**

**Points:** 100 (76 auto-graded + 24 manual)

---

## Learning Objectives

By the end of this in-class exercise you will be able to:

1. Turn a plausible range into a probability distribution, and say what that choice assumes.
2. Implement a Monte Carlo simulation in `numpy` to propagate input uncertainty through a cost model.
3. Check a simulated standard deviation against an analytical one, and explain the gap.
4. Read percentiles and exceedance probabilities off a simulated distribution to set a contingency.
5. Decompose output variance by input, and compare the result to the Week 12 tornado.

---

## Scenario: What Should the Contingency Be?

Week 12 ranked four inputs of a bid-item cost model at a 2,000 CY excavation job:

$$\text{item cost} = Q \times P \times (1 + \text{overhead}) \times (1 + \text{escalation})$$

| Input | Swing | % of base cost |
|---|---|---|
| Unit price | $59,144 | 117.2% |
| Quantity | $15,138 | 30.0% |
| Overhead | $4,465 | 8.8% |
| Escalation | $2,883 | 5.7% |

Base cost: **$50,459**. Pushing all four to their worst corner at once gave a
spread of **170.3%** of base — a number the in-class exercise told you not to budget against,
because all four going wrong together is far less likely than any one of them.

This week you find out how much less likely. An estimator has to name a
**contingency percentage**, and that is a probability statement. A tornado
cannot make one. Monte Carlo can.

## Before You Begin

**No data file to upload.** Every number in this in-class exercise is generated by the code
itself, from the model Week 11 fitted and Week 12 ranked.

> **In Colab:** Runtime → Run all to execute all cells in order. If you edit a
> parameter cell, re-run from that cell downward.

*NumPy RNG API note (ungraded): This in-class exercise uses the legacy API: `np.random.seed(42)` followed by `np.random.normal(...)`. The Week 14 in-class exercise and assignment use the newer API: `rng = np.random.default_rng(SEED)` followed by `rng.normal(...)`. Both produce the same distributions — the difference is in reproducibility and thread safety. The newer API is preferred for new code because each `rng` instance is independent, which makes parallel and multi-threaded Monte Carlo safer. The seeded results differ between the two APIs (same seed ≠ same draws), which is why this in-class exercise and the assignment use different seed values.*

## Setup

In [ ]:
# ── Identify your submission ─────────────────────────────────────────
# Fill both in before you run anything else. NETID is what matches your
# work to your student record — a blank NETID takes a 5-point deduction.
NAME  = ""   # e.g. "Jordan Reyes"
NETID = ""   # e.g. "jreyes"

print(f"NAME  = {NAME}")
print(f"NETID = {NETID}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

In [ ]:
# ── The Week 12 design point, carried forward unchanged ────────────────────────
Q0    = 2000          # design quantity, cubic yards
P0    = 21.2636       # fitted unit price at 2,000 CY, $/CY
OH0   = 0.13          # overhead, base case
ESC0  = 0.05          # escalation, base case

RMSE  = 0.2418        # the price model's residual spread, in log10 units

N     = 10_000        # number of Monte Carlo draws


def cost(q, p, oh, esc):
    """The same four-input cost model Week 12 ranked."""
    return q * p * (1 + oh) * (1 + esc)


BASE = cost(Q0, P0, OH0, ESC0)
print(f"base cost = ${BASE:,.0f}")

### Turning a Range into a Distribution

Week 12 gave each input a low and a high. A Monte Carlo needs more than that —
it needs a **distribution**, and choosing one is a claim about what you know.

Read each of Week 12's ranges as a **90% interval**: the plausible range covers
90% of cases, so the half-width is 1.645 standard deviations.

| Input | Week 12 range | This week's distribution |
|---|---|---|
| Quantity | ±15% | Normal(2,000, σ = 182 CY) |
| Overhead | 8–18% | Normal(0.13, σ = 0.0304) |
| Escalation | 2–8% | Normal(0.05, σ = 0.0182) |

**Unit price is different, and the difference matters.** Week 12 used ±1 RMSE —
the spread of what the whole market charges — because it had nothing better. An
estimator pricing a real job is not in that position: they have quotes. This in-class exercise
assumes **three quotes in hand**, a spread of about ±12%, so σ = 0.05 in log₁₀
units rather than the model's 0.2418.

B10 puts that assumption back and shows what it costs.

In [ ]:
Z90 = 1.645          # a +/-x% "plausible range", read as a 90% interval

SIG_Q    = Q0 * 0.15 / Z90     # +/-15% take-off
SIG_OH   = 0.05 / Z90          # 8% to 18%
SIG_ESC  = 0.03 / Z90          # 2% to 8%
SIG_LOGP = 0.05              # three quotes in hand, in log10 units

print(f"sigma_Q    = {SIG_Q:,.1f} CY")
print(f"sigma_OH   = {SIG_OH:.4f}")
print(f"sigma_Esc  = {SIG_ESC:.4f}")
print(f"sigma_logP = {SIG_LOGP}  ->  price band x{10**SIG_LOGP:.2f} / x{10**-SIG_LOGP:.2f}")
print(f"  (the model's own RMSE would be {RMSE} -> x{10**RMSE:.2f} / x{10**-RMSE:.2f})")

---

### A1 — Mean Item Cost, Quantity Only *(2 pts)*

Start with one input. Vary the take-off quantity alone, hold the other three at
their base values, and simulate `N` jobs.

In [ ]:
np.random.seed(42)
q_only = np.random.normal(Q0, SIG_Q, N)          # Normal(2000, 182), N draws

cost_qty = cost(q_only, P0, OH0, ESC0)

# Should land close to BASE, because the quantity draws are centred on Q0
ANSWER_A1_cost_mean_qty = round(___, 0)   # fill in: the mean of these draws
print(f"ANSWER_A1_cost_mean_qty = {ANSWER_A1_cost_mean_qty}")

### A2 — Standard Deviation, Quantity Only *(2 pts)*

In [ ]:
ANSWER_A2_cost_std_qty = round(___, 0)   # fill in: the spread of those draws
print(f"ANSWER_A2_cost_std_qty = {ANSWER_A2_cost_std_qty}")

### A3 — The Analytical Check *(2 pts)*

Cost is **linear** in quantity, so you do not need a simulation to know its
standard deviation:

$$\sigma_{\text{cost}} = \left|\frac{\partial\,\text{cost}}{\partial Q}\right| \times \sigma_Q
= P \times (1+\text{OH}) \times (1+\text{Esc}) \times \sigma_Q$$

Compute it, and compare against A2.

In [ ]:
sigma_analytical = SIG_Q * P0 * (1 + OH0) * (1 + ESC0)

ANSWER_A3_analytical_std = round(___, 0)   # fill in: the sigma computed above
print(f"ANSWER_A3_analytical_std = {ANSWER_A3_analytical_std}")
print(f"  simulated  (A2): ${ANSWER_A2_cost_std_qty:,.0f}")
print(f"  analytical (A3): ${ANSWER_A3_analytical_std:,.0f}")
print(f"  difference:      ${abs(ANSWER_A2_cost_std_qty - ANSWER_A3_analytical_std):,.0f}  (sampling noise)")

> **Why bother simulating at all, then?** Because the shortcut needs the model
> to be linear in that input and the input to be Normal. Unit price is neither —
> it enters through a log₁₀ fit, so its distribution is not Normal in dollars.
> That is where the simulation earns its keep.

---

## The Full Four-Input Monte Carlo

Now all four at once. Seed **once**, then draw each input in turn — the order
matters, because one seeded stream is handed out in the order you ask for it.

In [ ]:
np.random.seed(42)
q    = np.random.normal(Q0, SIG_Q, N)                           # quantity, CY
p    = 10 ** np.random.normal(np.log10(P0), SIG_LOGP, N)        # unit price, $/CY
oh   = np.random.normal(OH0, SIG_OH, N)                         # overhead
esc  = np.random.normal(ESC0, SIG_ESC, N)                       # escalation

item_cost = cost(q, p, oh, esc)

print(f"{N:,} simulated jobs")
print(f"  quantity   [{q.min():,.0f}, {q.max():,.0f}] CY")
print(f"  unit price [${p.min():.2f}, ${p.max():.2f}] per CY")
print(f"  item cost  [${item_cost.min():,.0f}, ${item_cost.max():,.0f}]")

### A4 — Mean Item Cost, Full MC *(2 pts)*

In [ ]:
ANSWER_A4_cost_mean_full = round(___, 0)   # fill in: mean of the four-input draws
print(f"ANSWER_A4_cost_mean_full = {ANSWER_A4_cost_mean_full}")
print(f"  base cost was ${BASE:,.0f}")

### A5 — Standard Deviation, Full MC *(2 pts)*

In [ ]:
ANSWER_A5_cost_std_full = round(___, 0)   # fill in: spread of the four-input draws
print(f"ANSWER_A5_cost_std_full = {ANSWER_A5_cost_std_full}")

### A6 — 5th Percentile *(2 pts)*

In [ ]:
ANSWER_A6_cost_p05 = round(___, 0)   # fill in: the 5th percentile of those draws
print(f"ANSWER_A6_cost_p05 = {ANSWER_A6_cost_p05}")

### A7 — 95th Percentile *(2 pts)*

In [ ]:
ANSWER_A7_cost_p95 = round(___, 0)   # fill in: the 95th percentile of those draws
print(f"ANSWER_A7_cost_p95 = {ANSWER_A7_cost_p95}")
print(f"  as a share of base: {ANSWER_A7_cost_p95 / BASE:.3f}")

### A8 — Probability of Exceeding \$60,000 *(2 pts)*

The estimator has carried **$60,000** in the bid — about 19% over the base cost.
What is the chance the item comes in above it?

In [ ]:
ANSWER_A8_prob_over_60k = round(___, 4)   # fill in: share of draws above $60,000
print(f"ANSWER_A8_prob_over_60k = {ANSWER_A8_prob_over_60k}")
print(f"  roughly 1 job in {1/ANSWER_A8_prob_over_60k:.0f}")

**The histogram.** Plot the simulated costs with the base cost and the 5th and
95th percentiles marked. This is what a tornado diagram cannot draw.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(item_cost, bins=60, color='#495A78', alpha=0.85)
ax.axvline(BASE, color='#0C234B', lw=2.5, ls='--', label=f'base ${BASE:,.0f}')
ax.axvline(ANSWER_A6_cost_p05, color='#AB0520', lw=2, label=f'P5  ${ANSWER_A6_cost_p05:,.0f}')
ax.axvline(ANSWER_A7_cost_p95, color='#AB0520', lw=2, label=f'P95 ${ANSWER_A7_cost_p95:,.0f}')
ax.set_xlabel('simulated item cost, dollars')
ax.set_ylabel('number of simulated jobs')
ax.set_title(f'{N:,} Monte Carlo draws — 2,000 CY excavation')
ax.legend()
plt.tight_layout()
plt.show()

---

### B1 — Why One Seed, Drawn in Order *(4 pts — manually graded)*

In 3–4 sentences, explain what `np.random.seed(42)` does and why the exercise seeds
**once** before drawing all four inputs rather than re-seeding before each one.
Then say what would go wrong if you re-seeded with 42 before each draw.

**B1 Answer (replace this text with your explanation):**

### B2–B4 — Variance Decomposition *(5 + 5 + 5 pts)*

The tornado ranked inputs by swing. A Monte Carlo can do better: it can say what
**share of the output variance** each input is responsible for.

Run the simulation four times, each with only one input random and the other
three fixed at their base values. Because the inputs are independent, the four
variances add.

In [ ]:
def var_from(which):
    """Variance of item cost when only `which` is random."""
    np.random.seed(42)
    a = dict(q=Q0, p=P0, oh=OH0, esc=ESC0)
    if which == 'q':   a['q']   = np.random.normal(Q0, SIG_Q, N)
    if which == 'p':   a['p']   = 10 ** np.random.normal(np.log10(P0), SIG_LOGP, N)
    if which == 'oh':  a['oh']  = np.random.normal(OH0, SIG_OH, N)
    if which == 'esc': a['esc'] = np.random.normal(ESC0, SIG_ESC, N)
    return cost(**a).var()

var_p, var_q  = var_from('p'), var_from('q')
var_oh, var_e = var_from('oh'), var_from('esc')
var_total = var_p + var_q + var_oh + var_e

ANSWER_B2_var_price_pct = round(___, 1)   # fill in: price's share of variance, %
ANSWER_B3_var_qty_pct   = round(___, 1)   # fill in: quantity's share, %
ANSWER_B4_var_oh_pct    = round(___, 2)   # fill in: overhead's share, %

print(f"ANSWER_B2_var_price_pct = {ANSWER_B2_var_price_pct}")
print(f"ANSWER_B3_var_qty_pct = {ANSWER_B3_var_qty_pct}")
print(f"ANSWER_B4_var_oh_pct = {ANSWER_B4_var_oh_pct}")
print(f"  escalation: {var_e / var_total * 100:.2f}%")
print(f"  the four shares sum to {(var_p+var_q+var_oh+var_e)/var_total*100:.1f}%")

In [ ]:
labels = ['Unit price', 'Quantity', 'Overhead', 'Escalation']
shares = [var_p, var_q, var_oh, var_e]
shares = [100 * s / var_total for s in shares]

fig, ax = plt.subplots(figsize=(9, 2.2))
left = 0
for label, s, c in zip(labels, shares, ['#0C234B', '#495A78', '#AB0520', '#F19E1F']):
    ax.barh(0, s, left=left, color=c, label=f'{label} {s:.1f}%')
    left += s
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('share of total variance in item cost (%)')
ax.set_title('Where the uncertainty comes from')
ax.legend(ncol=4, fontsize=9, loc='lower center', bbox_to_anchor=(0.5, -0.95))
plt.tight_layout()
plt.show()

### B5–B7 — The 80% Band *(5 + 5 + 5 + 5 pts)*

Percentiles are what an estimator can actually use. Report the 10th, the median
and the 90th, then the width of the band between them — in dollars and as a
percentage of the base cost.

In [ ]:
ANSWER_B5_cost_p10 = round(___, 0)   # fill in: the 10th percentile
ANSWER_B5_cost_p50 = round(___, 0)   # fill in: the median
ANSWER_B6_cost_p90 = round(___, 0)   # fill in: the 90th percentile

ANSWER_B7_cost_range_p80 = round(___, 0)   # fill in: the width of that band
ANSWER_B7_range_pct      = round(___, 1)   # fill in: that width as a % of BASE

print(f"ANSWER_B5_cost_p10 = {ANSWER_B5_cost_p10}")
print(f"ANSWER_B5_cost_p50 = {ANSWER_B5_cost_p50}")
print(f"ANSWER_B6_cost_p90 = {ANSWER_B6_cost_p90}")
print(f"ANSWER_B7_cost_range_p80 = {ANSWER_B7_cost_range_p80}")
print(f"ANSWER_B7_range_pct = {ANSWER_B7_range_pct}")
print()
print(f"  Week 12's all-corner spread was 170.3% of base.")
print(f"  This 80% band is {ANSWER_B7_range_pct}%.")

### B8 — Probability of Exceeding \$65,000 *(6 pts)*

In [ ]:
ANSWER_B8_prob_over_65k = round(___, 4)   # fill in: share of draws above $65,000
print(f"ANSWER_B8_prob_over_65k = {ANSWER_B8_prob_over_65k}")

### B9 — The Contingency *(6 pts)*

This is the number the week exists to produce. Find the **smallest contingency**,
in whole steps of 5%, that holds the chance of an overrun below **5%**.

In [ ]:
ANSWER_B9_contingency_pct = None

for pct in range(0, 105, 5):
    prob_over = (item_cost > BASE * (1 + pct / 100)).mean()
    print(f"  contingency {pct:>3}%  ->  P(overrun) = {prob_over:.4f}")
    if ___:                    # fill in: the overrun probability is below 5%
        ANSWER_B9_contingency_pct = pct
        break

print()
print(f"ANSWER_B9_contingency_pct = {ANSWER_B9_contingency_pct}")
print(f"  bid the item at ${BASE * (1 + ANSWER_B9_contingency_pct/100):,.0f}")

### B10 — What If You Had No Quotes? *(5 + 3 pts)*

Every number above rests on one assumption: that three quotes narrow the price to
σ = 0.05 in log₁₀ units. Put the model's own residual spread back — σ = RMSE =
0.2418, the whole market — and re-run.

In [ ]:
np.random.seed(42)
q_m   = np.random.normal(Q0, SIG_Q, N)
p_m   = 10 ** np.random.normal(np.log10(P0), RMSE, N)      # the only change
oh_m  = np.random.normal(OH0, SIG_OH, N)
esc_m = np.random.normal(ESC0, SIG_ESC, N)

cost_market = cost(q_m, p_m, oh_m, esc_m)

ANSWER_B10_market_std = round(___, 0)   # fill in: spread of the no-quotes draws
ANSWER_B10_std_ratio  = round(___, 2)   # fill in: how many times wider than A5

print(f"ANSWER_B10_market_std = {ANSWER_B10_market_std}")
print(f"ANSWER_B10_std_ratio = {ANSWER_B10_std_ratio}")
print()
print(f"  with quotes    mean ${item_cost.mean():,.0f}   median ${np.median(item_cost):,.0f}")
print(f"  without quotes mean ${cost_market.mean():,.0f}   median ${np.median(cost_market):,.0f}")

> **Look at the mean against the median.** With quotes they nearly coincide.
> Without them the mean sits well above the median, because a log-normal price
> has a long right tail — a few simulated jobs price enormously high and drag the
> average with them. The *typical* job did not get more expensive. The *average*
> did.

---

### B Written — The Tornado and the Decomposition *(4 pts — manually graded)*

In 4–6 sentences: compare your B2–B4 variance shares against Week 12's tornado
ranking. Do they agree on the order? Then explain why the shares are **not**
proportional to the swings — what does squaring do to a ranking? Close by saying
which of the two you would put in front of an estimating manager, and why.

**B Written Answer (replace this text):**

---

## Memo *(12 pts — manually graded)*

Write a short technical memo (To / From / Date / Subject, then three paragraphs)
to the estimating manager who asked what contingency to carry on this item.

- **Paragraph 1** — the recommendation. Cite B9, and state the risk it leaves
  open rather than implying there is none.
- **Paragraph 2** — why it is not 170%. Compare your 80% band (B7) against Week
  12's all-corner spread, and explain what corner stacking assumed.
- **Paragraph 3** — the assumption the number rests on. Use B10: the whole answer
  depends on having quotes. Say what you would do if the bid were due before they
  arrived.

**MEMO**

---

## Reflection *(4 pts — manually graded)*

In 100–150 words: Week 12 ranked inputs and Week 13 put a probability on the
answer. Describe one decision you could make with the tornado but not with the
Monte Carlo, and one you could make with the Monte Carlo but not with the
tornado. Support each with a number from this week.

**Reflection Answer (replace this text):**

---

In [ ]:
# ── Results Summary ─────────────────────────────────────────────────────────────
print("=" * 62)
print("CE 310 Week 13 — Monte Carlo Simulation Results Summary")
print("=" * 62)
print()
print("SECTION A — One Input, Then Four")
print(f"  A1  mean cost, quantity only : ${ANSWER_A1_cost_mean_qty:>12,.0f}")
print(f"  A2  std,       quantity only : ${ANSWER_A2_cost_std_qty:>12,.0f}")
print(f"  A3  std,       analytical    : ${ANSWER_A3_analytical_std:>12,.0f}")
print(f"  A4  mean cost, full MC       : ${ANSWER_A4_cost_mean_full:>12,.0f}")
print(f"  A5  std,       full MC       : ${ANSWER_A5_cost_std_full:>12,.0f}")
print(f"  A6  P5                       : ${ANSWER_A6_cost_p05:>12,.0f}")
print(f"  A7  P95                      : ${ANSWER_A7_cost_p95:>12,.0f}")
print(f"  A8  P(cost > $60,000)        : {ANSWER_A8_prob_over_60k:>13.4f}")
print()
print("SECTION B — Decomposition, Band and Contingency")
print(f"  B2  variance from unit price : {ANSWER_B2_var_price_pct:>12.1f}%")
print(f"  B3  variance from quantity   : {ANSWER_B3_var_qty_pct:>12.1f}%")
print(f"  B4  variance from overhead   : {ANSWER_B4_var_oh_pct:>12.2f}%")
print(f"  B5  P10                      : ${ANSWER_B5_cost_p10:>12,.0f}")
print(f"  B5  P50                      : ${ANSWER_B5_cost_p50:>12,.0f}")
print(f"  B6  P90                      : ${ANSWER_B6_cost_p90:>12,.0f}")
print(f"  B7  80% band width           : ${ANSWER_B7_cost_range_p80:>12,.0f}")
print(f"  B7  as % of base             : {ANSWER_B7_range_pct:>12.1f}%")
print(f"  B8  P(cost > $65,000)        : {ANSWER_B8_prob_over_65k:>13.4f}")
print(f"  B9  contingency for 5% risk  : {ANSWER_B9_contingency_pct:>12}%")
print(f"  B10 std, no quotes           : ${ANSWER_B10_market_std:>12,.0f}")
print(f"  B10 ratio to A5              : {ANSWER_B10_std_ratio:>12.2f}x")
print("=" * 62)

## Looking Ahead: Week 14 — Bootstrap Confidence Intervals

Monte Carlo sampled from distributions you **assumed** — Normal, log-normal.
Week 14 asks what to do when no assumption is defensible: resample the
**observed data** itself, and let it build a confidence interval around any
statistic at all — a slope, a percentile, an R², a ratio.


## Before You Submit

- [ ] `NAME` and `NETID` filled in at the top and showing in the cell output — a blank `NETID` takes a 5-point deduction
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `___` replaced
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Both charts plotted — the cost histogram and the variance decomposition bar
- [ ] B1, the B written answer, the memo and the reflection all filled in (not placeholder text)
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `CE310_W13_<NetID>.ipynb` and uploaded to the Week 13 D2L dropbox